# Story-Voice Fine-Tuner – Step 2: Prepare Data
**Gradio UI version**

We will:
- Upload your my_stories.txt file
- Load and clean the text
- Split into train/validation sets
- Tokenize for GPT-2-small
- Save the prepared dataset

In [1]:
from google.colab import files
import os

print("📤 Upload your writing samples file (my_stories.txt)")
uploaded = files.upload()

# Get the filename
txt_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {txt_filename}")

# Move it to the data folder (for consistency with our VS Code structure)
os.makedirs("/content/data", exist_ok=True)
os.rename(txt_filename, f"/content/data/{txt_filename}")
print("✅ File moved to /content/data/")

📤 Upload your writing samples file (my_stories.txt)


Saving my_stories.txt to my_stories.txt
✅ Uploaded: my_stories.txt
✅ File moved to /content/data/


In [2]:
# Load the entire text
with open(f"/content/data/{txt_filename}", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Basic cleaning (remove extra spaces, empty lines)
raw_text = "\n".join(line.strip() for line in raw_text.splitlines() if line.strip())

print(f"✅ Loaded {len(raw_text):,} characters of your writing")
print("First 300 characters preview:")
print(raw_text[:300])

✅ Loaded 228,736 characters of your writing
First 300 characters preview:
﻿Enduring forevermore
Enduring forevermore
On Solmara, in the newly occupied region of the wrath domain that birthed the budding territory of the youngest lord, Ashen Hart, a new department was developed.
The inquisition department.
The building lay a few meters away from the small church that the p


In [3]:
# Split: 90% for training, 10% for validation
split_index = int(len(raw_text) * 0.9)
train_text = raw_text[:split_index]
val_text   = raw_text[split_index:]

print(f"Train set: {len(train_text):,} characters")
print(f"Validation set: {len(val_text):,} characters")

Train set: 205,862 characters
Validation set: 22,874 characters


In [4]:
from transformers import GPT2Tokenizer
from datasets import Dataset

# Load the tokenizer (same one GPT-2 uses)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Important for training

# Create Hugging Face Datasets
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,      # GPT-2-small limit, good for beginners
        padding="max_length"
    )

# Convert text into dataset format
train_dataset = Dataset.from_dict({"text": [train_text]})
val_dataset   = Dataset.from_dict({"text": [val_text]})

# Tokenize
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val   = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("✅ Tokenization complete!")
print(f"Train tokens: {len(tokenized_train[0]['input_ids'])}")
print(f"Validation tokens: {len(tokenized_val[0]['input_ids'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

✅ Tokenization complete!
Train tokens: 512
Validation tokens: 512


In [5]:
# Save for Step 3 (and for VS Code later)
tokenized_train.save_to_disk("/content/data/tokenized_train")
tokenized_val.save_to_disk("/content/data/tokenized_val")

print("✅ Prepared datasets saved to /content/data/")

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1 [00:00<?, ? examples/s]

✅ Prepared datasets saved to /content/data/
